# QC confound 00 — Do ER+ and Normal specimens differ technically?

**Criticism being addressed.** PBS classes are thresholds on the same QC metrics that
ddqc shows shift with tissue and composition. ER+ tumours and reduction mammoplasties
also differ in handling, dissociation and sequencing depth, and Pal lowered the gene
threshold to 400/300 for low-coverage samples. A classifier on PBS features may be
predicting condition from those sample-level differences rather than from anything
special about discarded cells.

This series tests that alternative explanation directly:

| Notebook | Question |
|---|---|
| `00` specimen covariates | Do ER+ and Normal specimens differ in sample-level technical properties at all? |
| `01` negative controls | Does the same PBS pipeline applied to *retained* cells classify just as well? |
| `02` covariate adjustment | Does PBS composition still predict condition after adjusting for depth, complexity, dissociation stress and Pal's gene threshold? |
| `03` alternative noise definitions | Does the result survive when "discarded" is defined by one uniform or one adaptive rule instead of Pal's per-sample thresholds? |
| `04` depth equalization | Does the result survive when every library is thinned to the same sequencing depth? |

**This notebook** establishes the *necessary condition* for the confound: if ER+ and
Normal specimens do not differ in technical covariates, the confound cannot drive the
classifier. If they do (expected), notebooks 01–04 test whether it does.

One property of the published pipeline matters throughout: PBS quantile cutoffs are
recomputed **within each specimen's noise cells**. A pure shift of a specimen's QC
distribution therefore cancels out. What can still leak through is (a) *which* cells
are noise — set by Pal's per-sample thresholds — and (b) the *joint shape* of the three
metrics inside a specimen, which depth and dissociation can also change.

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from signals_in_the_noise.analysis.noise_phenotypes import ISCB_SCS_PBS_THRESHOLDS
from signals_in_the_noise.config import get_data_path
from signals_in_the_noise.preprocessing.gse161529 import GSE161529
from signals_in_the_noise.preprocessing.specimens import Cohort
from signals_in_the_noise.utils.logging_config import setup_logging

setup_logging(logging.INFO)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

POSITIVE, NEGATIVE = "ER+ tumour", "Normal"
SEED = 0
OUTPUT_DIR = get_data_path("processed") / "qc_confound"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
gse = GSE161529()
cohort = Cohort.from_objects(gse.objects.values(), [POSITIVE, NEGATIVE])
cohort.overview().groupby("condition").agg(specimens=("n_cells", "size"), cells=("n_cells", "sum"))

## 1. Specimens and patients

Specimens are keyed by GEO file name, because GSE161529 titles are **not unique**:
patient 0029 has two ER+ Total samples with the same title. The published notebook
(`iscb-scs/10-a`) keyed specimens by title, so one of them silently replaced the other.

Some patients contribute several specimens. Every classifier in this series holds out
**whole patients** so no patient is ever on both sides of a split; metrics are still
computed per specimen, because at least one patient (0064) has both a Normal and an
ER+ specimen.

In [ ]:
cohort.shared_patients()

In [ ]:
from signals_in_the_noise.modeling.datasets import build_cell_table, specimen_sizes, standard_arms
from signals_in_the_noise.modeling.experiments import PUBLISHED_PROTOCOL, reference_specimens

reference_arm = standard_arms(ISCB_SCS_PBS_THRESHOLDS, names=["noise_pbs"])[0]
reference_table = build_cell_table(
    cohort.obs, cohort.conditions, reference_arm, patients=cohort.patients
)
included = reference_specimens(reference_table, PUBLISHED_PROTOCOL["min_reference_cells"])
min_cells = PUBLISHED_PROTOCOL["min_reference_cells"]
print(f"{len(included)} specimens have >= {min_cells} PBS-labelled noise cells")

## 2. Technical covariates per specimen

`included` marks the specimens that enter the classifier under the published filter
(>= 50 PBS-labelled noise cells). Association tests are reported for the included
specimens (the cohort the classifier actually sees) and for all specimens.

`auc` is P(ER+ > Normal): 0.5 means no separation, 0 or 1 means perfect separation.

In [ ]:
from signals_in_the_noise.analysis.confounders import (
    PRESPECIFIED_TECHNICAL_COVARIATES,
    association_table,
    specimen_covariates,
)

covariates = specimen_covariates(cohort.specimens)
covariates["n_pbs_noise_cells"] = specimen_sizes(reference_table).reindex(
    covariates.index, fill_value=0
)
covariates["included"] = covariates.index.isin(included).astype(int)
tested = [c for c in covariates.select_dtypes("number").columns if c != "included"]

display(covariates.groupby("condition")["included"].agg(specimens="size", included="sum"))
covariates

In [ ]:
association_included = association_table(
    covariates[covariates["included"] == 1], positive=POSITIVE, negative=NEGATIVE, columns=tested
)
association_all = association_table(
    covariates, positive=POSITIVE, negative=NEGATIVE, columns=tested
)
print("Included specimens")
display(association_included.round(4))
print("All specimens")
display(association_all.round(4))

### 2.1 Pal's lowered gene threshold

`qc_genes_lower` below the study-wide mode marks the low-coverage adjustment. If it is
concentrated in one condition, the *definition* of noise differs by condition.

In [ ]:
for label, subset in [("included", covariates[covariates["included"] == 1]), ("all", covariates)]:
    print(label)
    display(pd.crosstab(subset["condition"], subset["qc_genes_lower"], margins=True))

In [ ]:
from signals_in_the_noise.utils.visualization import plot_covariates_by_condition

shown = [
    *PRESPECIFIED_TECHNICAL_COVARIATES,
    "noise_fraction",
    "n_pbs_noise_cells",
    "median_log1p_total_counts_noise",
    "median_pct_counts_mt_noise",
]
plot_covariates_by_condition(
    covariates[covariates["included"] == 1], shown, association=association_included
)
plt.show()

## 3. QC distributions per specimen

Location differences between specimens are expected; they cancel inside the PBS
quantiles. Look instead for condition-specific differences in **spread and shape**
(for example, a heavier low-count tail in one condition), which do not cancel.

In [ ]:
from signals_in_the_noise.preprocessing.qc import QC_METRICS
from signals_in_the_noise.utils.visualization import plot_specimen_qc_distributions

included_obs = {sid: cohort.obs[sid] for sid in included}
for population in ("noise", "retained"):
    plot_specimen_qc_distributions(
        included_obs, cohort.conditions, list(QC_METRICS), population=population
    )
    plt.show()

## 4. Can technical covariates alone predict condition?

Specimen-level logistic regression, patients held out, significance by permuting
condition labels across specimens. If technical covariates alone separate ER+ from
Normal, a PBS classifier *could* be exploiting them; notebooks 01–04 test whether it is.

In [ ]:
from signals_in_the_noise.modeling.evaluation import logistic_regression_factory
from signals_in_the_noise.modeling.experiments import compare_feature_sets

N_PERMUTATIONS = 1000
technical_sets = {"prespecified technical": (list(PRESPECIFIED_TECHNICAL_COVARIATES), None)}
technical_sets |= {c: ([c], None) for c in [*PRESPECIFIED_TECHNICAL_COVARIATES, "noise_fraction"]}

technical_only = compare_feature_sets(
    covariates.loc[included],
    technical_sets,
    positive_label=POSITIVE,
    n_permutations=N_PERMUTATIONS,
    model_factory=logistic_regression_factory(C=1.0),
    group="patient_id",
    seed=SEED,
)
technical_only

In [ ]:
covariates.to_csv(OUTPUT_DIR / "00_specimen_covariates.csv")
association_included.to_csv(OUTPUT_DIR / "00_association_included.csv")
association_all.to_csv(OUTPUT_DIR / "00_association_all.csv")
technical_only.to_csv(OUTPUT_DIR / "00_technical_only_classifier.csv")

## Reading the results

| Observation | Meaning |
|---|---|
| No covariate separates conditions (AUC near 0.5, q > 0.1) and the technical-only classifier is at chance | The confound has nothing to act on; the remaining notebooks are confirmatory. |
| Depth, complexity, mito or the gene threshold separate conditions | The confound is **possible**. Whether it explains the PBS classifier is decided in 01–04. |
| `lowered_genes_lower` concentrated in one condition | Noise membership itself differs by condition; notebook 03 is the decisive test. |